<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_14_file_io_json/note_lesson_14_file_io_json.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 14 — Файли та JSON: місячний звіт каси кафе

Наприкінці липня каса вивантажила файл `kasa_2024_07.txt` — ті самі рядки, що в уроці 13, зокрема зіпсовані. Сьогодні програма:

1. прочитає файл каси;
2. збереже звіт у `report_2024_07.json` — його відкриє сайт кафе чи бухгалтер;
3. запише зіпсовані рядки в `errors_2024_07.txt` — для касира;
4. допише рядок у журнал запусків `runs.log`.

Перші клітинки через `%%writefile` створюють файли, які в справжньому кафе дала б каса. У Colab вони з'являться на панелі 📁 ліворуч. Виконуй клітинки **зверху вниз**; перед клітинками з позначкою **Прогноз** спершу скажи, що буде.

Теорія — у книзі: [Урок 14. Файли, менеджери контексту та JSON](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m1/lesson_14/).

## 🔁 Пригадай (без підглядання)

1. Що повертає `load_orders(lines)` з уроку 13?
2. Програма завершилась. Де тепер її словник `report`?
3. Що дасть цикл `for` по ітератору, який уже вичерпали?

<details>
<summary>Відповіді</summary>

1. Пару: список прийнятих `RawOrder` і список пояснень до пропущених рядків.
2. Ніде: змінні живуть у пам'яті процесу і зникають разом з ним.
3. Нічого. Відкритий файл поводиться так само.

</details>

## 0. Файли, які дала каса

In [ ]:
%%writefile kasa_2024_07.txt
2024-07-19 18:30;540.00;50;2
2024-07-19 12:10;320.00;30;1
2024-07-19 19:05;540,00;40;3
2024-07-20 20:15;980.00;120;4
2024-07-20 13:40;760.00
2024-02-30 19:00;450.00;0;5
2024-07-21 18:00;-120.00;0;2
2024-07-21 14:20;610.00;60;0
2024-07-21 21:30;1200.00;150;6

In [ ]:
%%writefile cafe_config.json
{
  "cafe": "Кафе «Смачно»",
  "currency": "грн",
  "kasa_file": "kasa_{year}_{month:02d}.txt",
  "report_file": "report_{year}_{month:02d}.json",
  "errors_file": "errors_{year}_{month:02d}.txt",
  "log_file": "runs.log"
}

In [ ]:
%%writefile orders.csv
order_id,customer_name,dish,price,order_date,city
1,Anna,Pizza,320,2026-03-10 12:30,Kyiv
2,Oleh,Burger,210,2026-03-10 13:10,Lviv
3,Ira,Pasta,280,2026-03-11 18:45,Kyiv
4,Max,Soup,150,2026-03-11 14:20,Odesa
5,Nina,Pizza,320,2026-03-12 19:05,Lviv
6,Andrii,Salad,190,2026-03-12 11:50,Kyiv
7,Sofi,Burger,210,2026-03-13 16:40,Odesa
8,Den,Pasta,280,2026-03-13 20:15,Kyiv
9,Kate,Pizza,320,2026-03-14 13:00,Lviv
10,Vasyl,Soup,150,2026-03-14 19:30,Kyiv
11,Lena,Salad,190,2026-03-15 12:10,Odesa
12,Roman,Pasta,280,2026-03-15 20:45,Kyiv

## 1. Читаємо файл каси

`with open(шлях, режим, encoding="utf-8") as file:` — відкрити файл і **гарантовано** закрити після блоку. `read()` повертає весь текст одним рядком.

In [ ]:
with open("kasa_2024_07.txt", "r", encoding="utf-8") as file:
    text = file.read()

print(type(text), len(text))
print(text.splitlines()[0])

**Прогноз:** як виглядатимуть рядки, якщо читати файл циклом `for`? Зверни увагу на кінець рядка.

In [ ]:
with open("kasa_2024_07.txt", encoding="utf-8") as file:
    for number, line in enumerate(file, start=1):
        if number <= 2:
            print(repr(line))

<details>
<summary>Відповідь</summary>

`'2024-07-19 18:30;540.00;50;2\n'` — у кожного рядка в кінці символ переходу `\n`. Його прибирають `strip()`.

</details>

In [ ]:
def read_kasa(path):
    """Непорожні рядки файлу каси без символу нового рядка."""
    lines = []
    with open(path, encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                lines.append(line)
    return lines


lines = read_kasa("kasa_2024_07.txt")
print(len(lines), lines[-1])

### Навіщо `with`

**Прогноз:** чи буде файл закритий після винятку всередині `with`?

In [ ]:
with open("kasa_2024_07.txt", encoding="utf-8") as file:
    first = file.readline()

print(file.closed)


try:
    with open("kasa_2024_07.txt", encoding="utf-8") as file:
        first = file.readline()
        average = 860.0 / 0
except ZeroDivisionError:
    print("виняток усередині with")

print(file.closed)

<details>
<summary>Відповідь</summary>

Так, `True` обидва рази. `with` закриває файл за будь-якого виходу з блоку — як `finally` з уроку 13.

</details>

## 2. Розбір рядків — код з уроку 13 без змін

In [ ]:
from datetime import datetime
from typing import NamedTuple


class RawOrder(NamedTuple):
    total_bill: float
    tip: float
    size: int
    timestamp: datetime


def parse_line(line):
    """Рядок каси -> RawOrder. Зіпсований рядок -> ValueError з поясненням."""
    fields = line.split(";")
    if len(fields) != 4:
        raise ValueError(f"очікували 4 поля, а маємо {len(fields)}")
    time_text, bill_text, tip_text, size_text = fields
    timestamp = datetime.strptime(time_text, "%Y-%m-%d %H:%M")
    bill, tip, size = float(bill_text), float(tip_text), int(size_text)
    if bill <= 0:
        raise ValueError(f"сума чека має бути більшою за 0, а маємо {bill}")
    if tip < 0:
        raise ValueError(f"чайові не можуть бути від'ємними: {tip}")
    if size < 1:
        raise ValueError(f"гостей має бути хоча б один, а маємо {size}")
    return RawOrder(bill, tip, size, timestamp)


def load_orders(lines):
    """Правильні чеки і список пояснень до пропущених рядків."""
    orders = []
    errors = []
    for number, line in enumerate(lines, start=1):
        try:
            orders.append(parse_line(line))
        except ValueError as error:
            errors.append(f"рядок {number}: {error}")
    return orders, errors

In [ ]:
orders, errors = load_orders(read_kasa("kasa_2024_07.txt"))
print(len(orders), len(errors))

## 3. Режими `w` і `a`

`"w"` стирає файл і пише з нуля — для переліку помилок за місяць. `write` **не додає** `\n` сам.

In [ ]:
with open("errors_2024_07.txt", "w", encoding="utf-8") as file:
    for message in errors:
        file.write(message + "\n")

with open("errors_2024_07.txt", encoding="utf-8") as file:
    print(file.read(), end="")

`"a"` дописує в кінець — для журналу запусків.

**Прогноз:** запусти клітинку нижче, а потім **ще раз** закоментуй рядок з `unlink` і запусти знову. Скільки рядків буде в журналі?

In [ ]:
from pathlib import Path

Path("runs.log").unlink(missing_ok=True)   # починаємо з чистого журналу


def log_run(message, path="runs.log"):
    with open(path, "a", encoding="utf-8") as file:
        print(message, file=file)


log_run("2024-07: прийнято 4, пропущено 5")
log_run("2024-07: прийнято 4, пропущено 5")

with open("runs.log", encoding="utf-8") as file:
    print(file.read(), end="")

<details>
<summary>Відповідь</summary>

Перший запуск: 2 рядки. Другий (без `unlink`): 4 — `"a"` нічого не стирає. З `"w"` завжди лишався б один рядок.

</details>

## 4. Шляхи і `FileNotFoundError`

Відносний шлях шукається від **поточної теки** — у ноутбуці це тека ноутбука, у Colab — `/content`.

In [ ]:
print(Path.cwd())

In [ ]:
path = Path("kasa_2024_08.txt")
print(path.exists())

try:
    read_kasa(path)
except FileNotFoundError as error:
    print("Каса ще не вивантажила файл:", error.filename)


folder = Path("kasa")
path = folder / "kasa_2024_07.txt"
print(path.name, path.stem, path.suffix)
print(path.parent.name)

## 5. JSON

**Прогноз:** що повернеться з файлу, записаного через `str(report)`?

In [ ]:
report = {"cafe": "Смачно", "orders": 4, "revenue": 3040.0, "open": True, "note": None}

with open("report.txt", "w", encoding="utf-8") as file:
    file.write(str(report))

with open("report.txt", encoding="utf-8") as file:
    back = file.read()

print(type(back))
print(back[:12])

<details>
<summary>Відповідь</summary>

Рядок (`<class 'str'>`), а не словник: `back["orders"]` не спрацює. Тому — JSON.

</details>

In [ ]:
import json

text = json.dumps(report, ensure_ascii=False)
print(text)
print(json.loads(text) == report)

print(json.dumps({"cafe": "Смачно"}))

### Що JSON вміє, а що ні

**Прогноз:** що зміниться після `dumps` → `loads`? Кортеж, ключ `7`, `Counter`.

In [ ]:
from collections import Counter

data = {"best_day": ("сб", 980.0), 7: "липень", "by_time": Counter({"вечеря": 3, "обід": 1})}
back = json.loads(json.dumps(data, ensure_ascii=False))
print(back)

<details>
<summary>Відповідь</summary>

Кортеж → список, ключ `7` → рядок `'7'`, `Counter` → звичайний словник. Помилки немає, але `back[7]` дасть `KeyError`.

</details>

`datetime` JSON не знає. Розкоментуй перший рядок — побачиш `TypeError`, потім закоментуй назад. Дату пишуть рядком ISO:

In [ ]:
# json.dumps({"first_order": datetime(2024, 7, 19, 12, 10)})

stamp = datetime(2024, 7, 19, 12, 10).isoformat()
print(stamp)
print(datetime.fromisoformat(stamp))

In [ ]:
try:
    json.loads("{'orders': 4}")
except json.JSONDecodeError as error:
    print(isinstance(error, ValueError))
    print(error)

## 6. Налаштування у файлі

`cafe_config.json` тримає назву кафе й шаблони імен файлів. `str.format` підставляє значення в шаблон з файлу.

In [ ]:
with open("cafe_config.json", encoding="utf-8") as file:
    config = json.load(file)

print(config["cafe"])
print(config["kasa_file"].format(year=2024, month=7))

## 7. CSV

`csv.DictReader` віддає кожен рядок словником. **Усі значення — рядки**, числа перетворюємо самі.

In [ ]:
import csv

with open("orders.csv", encoding="utf-8", newline="") as file:
    rows = list(csv.DictReader(file))

print(len(rows))
print(rows[0]["dish"], repr(rows[0]["price"]))
print(sum(float(row["price"]) for row in rows))

## 8. Розібраний приклад: місячний звіт

`build_report` лише рахує; `month_report` читає й пише файли; `main` розмовляє з людиною.

In [ ]:
DAYS = ("пн", "вт", "ср", "чт", "пт", "сб", "нд")


def meal_type_from_hour(hour):
    if 11 <= hour <= 15:
        return "обід"
    if 17 <= hour <= 23:
        return "вечеря"
    return "інше"

In [ ]:
def build_report(config, period, orders, errors):
    """Звіт за місяць — лише типи, які розуміє JSON."""
    revenue = sum(order.total_bill for order in orders)
    average = revenue / len(orders) if orders else 0.0
    by_time = Counter(meal_type_from_hour(order.timestamp.hour) for order in orders)
    first = min(order.timestamp for order in orders).isoformat() if orders else None
    return {
        "cafe": config["cafe"],
        "currency": config["currency"],
        "period": period,
        "orders": len(orders),
        "skipped": len(errors),
        "revenue": round(revenue, 2),
        "average": round(average, 2),
        "by_time": dict(by_time.most_common()),
        "first_order": first,
    }


def month_report(config, year, month):
    names = {key: config[key].format(year=year, month=month)
             for key in ("kasa_file", "report_file", "errors_file")}
    orders, errors = load_orders(read_kasa(names["kasa_file"]))
    report = build_report(config, f"{year}-{month:02d}", orders, errors)

    with open(names["report_file"], "w", encoding="utf-8") as file:
        json.dump(report, file, ensure_ascii=False, indent=2)
    with open(names["errors_file"], "w", encoding="utf-8") as file:
        for message in errors:
            print(message, file=file)
    log_run(f"{report['period']}: прийнято {report['orders']}, пропущено {report['skipped']}",
            config["log_file"])
    return names["report_file"]


def main(year, month):
    with open("cafe_config.json", encoding="utf-8") as file:
        config = json.load(file)
    try:
        report_file = month_report(config, year, month)
    except FileNotFoundError as error:
        print("Немає файлу каси:", error.filename)
        return 1
    print("Звіт збережено:", report_file)
    return 0


main(2024, 7)
main(2024, 8)

In [ ]:
with open("report_2024_07.json", encoding="utf-8") as file:
    print(file.read())


with open("report_2024_07.json", encoding="utf-8") as file:
    saved = json.load(file)

print(f"{saved['cafe']}, {saved['period']}: {saved['revenue']:.2f} {saved['currency']}")
print("Вечері:", saved["by_time"]["вечеря"])

## 🛠 Вправа 1. Найкращий день і чайові

Напиши `extra_stats(orders)`, що повертає пару `(best_day, tips_percent)`:

- `best_day` — день тижня (`DAYS[order.timestamp.weekday()]`) з найбільшим виторгом, `None` для порожнього списку;
- `tips_percent` — чайові як відсоток від виторгу, округлені до одного знака; `0.0` для порожнього списку.

Потім додай ці два ключі в `build_report` і перезапусти `main(2024, 7)`.

In [ ]:
def extra_stats(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    revenue_by_day = {}
    for order in orders:
        day = DAYS[order.timestamp.weekday()]
        revenue_by_day[day] = revenue_by_day.get(day, 0) + order.total_bill
    if not revenue_by_day:
        return None, 0.0
    best_day = max(revenue_by_day, key=revenue_by_day.get)
    revenue = sum(revenue_by_day.values())
    tips = sum(order.tip for order in orders)
    return best_day, round(tips / revenue * 100, 1)
    # END SOLUTION


print(extra_stats(orders))
assert extra_stats(orders) == ("нд", 11.5)
assert extra_stats([]) == (None, 0.0)
print("✅ Вправа 1 пройдена")

## 🛠 Вправа 2. Картка постійного гостя

Бали лояльності зберігаються між запусками в `loyalty.json`.

- `load_loyalty(path)` — словник з файлу; файлу ще немає → `{}` (через `try` / `except FileNotFoundError`);
- `add_visit(cards, name)` — +1 бал, новому гостю — картка з 1 балом; файлів не торкається;
- `save_loyalty(cards, path)` — запис кирилицею з відступами.

Перевірка імітує два запуски програми.

In [ ]:
def load_loyalty(path):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        with open(path, encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        return {}
    # END SOLUTION


def add_visit(cards, name):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    cards[name] = cards.get(name, 0) + 1
    # END SOLUTION


def save_loyalty(cards, path):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    with open(path, "w", encoding="utf-8") as file:
        json.dump(cards, file, ensure_ascii=False, indent=2)
    # END SOLUTION


Path("loyalty.json").unlink(missing_ok=True)

cards = load_loyalty("loyalty.json")          # перший запуск
assert cards == {}
for name in ["Оксана", "Тарас", "Оксана"]:
    add_visit(cards, name)
save_loyalty(cards, "loyalty.json")

cards = load_loyalty("loyalty.json")          # другий запуск
assert cards == {"Оксана": 2, "Тарас": 1}
add_visit(cards, "Оксана")
save_loyalty(cards, "loyalty.json")

assert load_loyalty("loyalty.json") == {"Оксана": 3, "Тарас": 1}
assert "Оксана" in Path("loyalty.json").read_text(encoding="utf-8")
print("✅ Вправа 2 пройдена")

## 🛠 Вправа 3. Знайди помилку

У кожній клітинці одна помилка. Виправ так, щоб `assert` пройшов.

In [ ]:
# Баг 1: журнал після двох запусків має містити два рядки
Path("runs.log").unlink(missing_ok=True)
for _ in range(2):
    with open("runs.log", "w", encoding="utf-8") as file:
        file.write("2024-07: прийнято 4, пропущено 5")
# BEGIN SOLUTION
Path("runs.log").unlink(missing_ok=True)
for _ in range(2):
    with open("runs.log", "a", encoding="utf-8") as file:
        file.write("2024-07: прийнято 4, пропущено 5\n")
# END SOLUTION
assert Path("runs.log").read_text(encoding="utf-8").splitlines() == ["2024-07: прийнято 4, пропущено 5"] * 2
print("✅ Баг 1 виправлено")

In [ ]:
# Баг 2: у файлі має бути по повідомленню в рядку
with open("errors_2024_07.txt", "w", encoding="utf-8") as file:
    for message in errors:
        file.write(message)
# BEGIN SOLUTION
with open("errors_2024_07.txt", "w", encoding="utf-8") as file:
    for message in errors:
        file.write(message + "\n")
# END SOLUTION
assert len(Path("errors_2024_07.txt").read_text(encoding="utf-8").splitlines()) == 5
print("✅ Баг 2 виправлено")

In [ ]:
# Баг 3: розкоментуй — чому TypeError?
# with open("report_2024_07.json", encoding="utf-8") as file:
#     saved = json.loads(file)
# BEGIN SOLUTION
with open("report_2024_07.json", encoding="utf-8") as file:
    saved = json.load(file)
# END SOLUTION
assert saved["orders"] == 4
print("✅ Баг 3 виправлено")

<details>
<summary>Пояснення</summary>

1. `"w"` стирає журнал щоразу, і немає `\n`. 2. `write` не додає `\n`. 3. `json.loads` чекає рядок; для файлу — `json.load`.

</details>

## ✅ Самоперевірка

1. Чим `with open(...)` кращий за `open(...)` + `close()`?
2. Файл відкрили в `"w"` і нічого не записали. Що з вмістом?
3. Чому другий `file.read()` поспіль повертає `""`?
4. Чому звіт зберігають у JSON, а не через `str(report)`?
5. У звіті був ключ `7`. Як дістати значення після `json.load`?
6. Чим `json.load` відрізняється від `json.loads`?

<details>
<summary>Відповіді</summary>

1. `with` закриває файл за будь-якого виходу з блоку, зокрема після винятку.
2. Стерто: `"w"` очищає файл під час відкриття.
3. Курсор уже в кінці файлу; треба відкрити заново або `file.seek(0)`.
4. `str()` не збирається назад у словник і незрозумілий іншим мовам; JSON — так.
5. `saved["7"]`: ключі JSON — лише рядки.
6. `load` — з файлу, `loads` — з рядка.

</details>

### Шпаргалка

```python
with open(path, "r", encoding="utf-8") as file:   # "w" — стерти й писати, "a" — дописати
    text = file.read()                            # усе одним рядком
    for line in file: line.strip()                # або рядок за рядком

print(message, file=file)                         # запис з \n
file.write(message + "\n")                        # write сам \n не додає

from pathlib import Path
Path("kasa") / "kasa_2024_07.txt"; path.exists(); Path.cwd()

import json
json.dump(obj, file, ensure_ascii=False, indent=2); json.load(file)
json.dumps(obj); json.loads(text)                 # з рядками
datetime.isoformat() / datetime.fromisoformat(text)

import csv
csv.DictReader(file)                              # рядки-словники, значення — str
```

## Далі

**Урок 15 — Git + GitHub.** Файли проєкту кафе вже є — час зберігати їхню історію й показувати код іншим.

Довідник — [`notes_file_io_json.ipynb`](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_1/lessons/lesson_14_file_io_json/notes_file_io_json.ipynb): курсор файлу, типи JSON докладно, телефонна книга.